In [1]:
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
import numpy as np
from sklearn.metrics import f1_score
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
train_df = pd.read_parquet("/content/drive/MyDrive/Colab Notebooks/NLP/train.parquet")
val_df = pd.read_parquet("/content/drive/MyDrive/Colab Notebooks/NLP/validation.parquet")

langs = ["ko", "ar", "te"]
train_datasets, val_datasets = {}, {}

for lang in langs:
    train_datasets[lang] = Dataset.from_pandas(train_df[train_df["lang"] == lang])
    val_datasets[lang] = Dataset.from_pandas(val_df[val_df["lang"] == lang])

train_dataset_ko, val_dataset_ko = train_datasets["ko"], val_datasets["ko"]
train_dataset_ar, val_dataset_ar = train_datasets["ar"], val_datasets["ar"]
train_dataset_te, val_dataset_te = train_datasets["te"], val_datasets["te"]

## **Initialization & Encoding**

In [3]:
# Initialize
tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-base')
model = AutoModelForTokenClassification.from_pretrained(
    'xlm-roberta-base',
    num_labels=3,
    id2label={0: 'O', 1: 'B-ANS', 2: 'I-ANS'},
    label2id={'O': 0, 'B-ANS': 1, 'I-ANS': 2}
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## **Prepocessing**

In [5]:
def preprocess(examples):
    tokenized = tokenizer(
        examples['question'],
        examples['context'],
        truncation='only_second',
        max_length=512,
        padding='max_length',
        return_offsets_mapping=True
    )

    labels_batch = []
    for i, offsets in enumerate(tokenized['offset_mapping']):
        labels = [-100] * len(offsets)
        if examples['answerable'][i] and examples['answer_start'][i] is not None:
            a_start, a_end = int(examples['answer_start'][i]), int(examples['answer_start'][i]) + len(examples['answer'][i])
            first = True
            for j, (s, e) in enumerate(offsets):
                if s == 0 and e == 0:
                    continue
                if s < a_end and e > a_start:
                    labels[j] = 1 if first else 2
                    first = False
                elif e > a_end:
                    break
        labels_batch.append(labels)

    tokenized['labels'] = labels_batch
    return tokenized

In [6]:
# preprocess and evaluation function for validation data
def preprocess_val(dataset):
    dataset = dataset.map(preprocess, batched=True, remove_columns=dataset.column_names)
    return dataset.filter(lambda x: any(l != -100 for l in x["labels"]))

def f1(eval_dataset,trainer):
    predictions, labels, _ = trainer.predict(eval_dataset)
    preds = np.argmax(predictions, axis=-1)
    mask = labels != -100
    return f1_score(labels[mask], preds[mask], average="macro")

## **Arabic Model**

In [7]:
train_dataset = train_dataset_ar.map(preprocess, batched=True)
val_dataset = val_dataset_ar.map(preprocess, batched=True)

Map:   0%|          | 0/2558 [00:00<?, ? examples/s]

Map:   0%|          | 0/415 [00:00<?, ? examples/s]

In [10]:
training_args = TrainingArguments(
    output_dir='./results',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=1,
    eval_strategy='steps',
    load_best_model_at_end=True,
    logging_steps=50,
    eval_steps=200,                 # evaluate less frequently than logging
    save_strategy="steps",
    save_steps=200,                 # save model every eval
    metric_for_best_model="loss",   # pick metric to judge best model
    greater_is_better=False,
    report_to="none"
)

trainer_ar = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=DataCollatorForTokenClassification(tokenizer),
)


In [11]:
trainer_ar.train()

Step,Training Loss,Validation Loss


TrainOutput(global_step=160, training_loss=0.242355015873909, metrics={'train_runtime': 56.2418, 'train_samples_per_second': 45.482, 'train_steps_per_second': 2.845, 'total_flos': 668403146668032.0, 'train_loss': 0.242355015873909, 'epoch': 1.0})

In [15]:
val_ar_dataset = preprocess_val(val_dataset_ar)

Map:   0%|          | 0/415 [00:00<?, ? examples/s]

Filter:   0%|          | 0/415 [00:00<?, ? examples/s]

In [16]:
f1_ar = f1(val_dataset,trainer_ar)
print("Token-level F1 per language:")
print(f"Arabic: {f1_ar:.4f}")

NameError: name 'trainer' is not defined

epochs=5: 0.8447

## **Korean Model** (restart sessison)

In [ ]:
train_dataset = train_dataset_ko.map(preprocess, batched=True)
val_dataset = val_dataset_ko.map(preprocess, batched=True)

Map:   0%|          | 0/2422 [00:00<?, ? examples/s]

Map:   0%|          | 0/356 [00:00<?, ? examples/s]

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=5,
    eval_strategy='steps',
    load_best_model_at_end=True,
    logging_steps=50,
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    metric_for_best_model="loss",
    greater_is_better=False,
    report_to="none"
)

trainer_ko = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=DataCollatorForTokenClassification(tokenizer),
)


In [ ]:
trainer_ko.train()

Step,Training Loss,Validation Loss
200,0.198500,0.157716
400,0.146000,0.152589
600,0.127900,0.161450


TrainOutput(global_step=760, training_loss=0.17635961598471592, metrics={'train_runtime': 285.4896, 'train_samples_per_second': 42.418, 'train_steps_per_second': 2.662, 'total_flos': 3164332332349440.0, 'train_loss': 0.17635961598471592, 'epoch': 5.0})

In [ ]:
val_ko_dataset = preprocess_val(val_dataset_ko)

Map:   0%|          | 0/356 [00:00<?, ? examples/s]

Filter:   0%|          | 0/356 [00:00<?, ? examples/s]

In [ ]:
f1_ko = f1(val_ko_dataset,trainer_ko)
print("Token-level F1 per language:")
print(f"Korean: {f1_ko:.4f}")


Token-level F1 per language:
Korean: 0.8943


epochs size = 3: 0.8870; epochs size = 5, batch=8: 0.8865; epochs size = 5, batch=16: 0.8943

##**Telugu Model**

In [ ]:
train_dataset = train_dataset_te.map(preprocess, batched=True)
val_dataset = val_dataset_te.map(preprocess, batched=True)

Map:   0%|          | 0/1355 [00:00<?, ? examples/s]

Map:   0%|          | 0/384 [00:00<?, ? examples/s]

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=8,
    eval_strategy='steps',
    load_best_model_at_end=True,
    logging_steps=50,
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    metric_for_best_model="loss",
    greater_is_better=False,
    report_to="none"
)

trainer_te = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=DataCollatorForTokenClassification(tokenizer),
)


In [ ]:
trainer_te.train()

Step,Training Loss,Validation Loss
200,0.278800,nan
400,0.169200,nan
600,0.142900,nan
800,0.112800,nan
1000,0.094200,nan
1200,0.067600,nan


TrainOutput(global_step=1360, training_loss=0.16057011055595735, metrics={'train_runtime': 300.6612, 'train_samples_per_second': 36.054, 'train_steps_per_second': 4.523, 'total_flos': 2832482451087360.0, 'train_loss': 0.16057011055595735, 'epoch': 8.0})

In [ ]:
val_te_dataset = preprocess_val(val_te_dataset)

Map:   0%|          | 0/384 [00:00<?, ? examples/s]

Filter:   0%|          | 0/384 [00:00<?, ? examples/s]

In [ ]:
f1_te = f1(val_dataset_te)

print("Token-level F1 per language:")
print(f"Telugu: {f1_te:.4f}")


Token-level F1 per language:
Telugu: 0.8494


In [ ]:
epochs=5: 0.8619; epochs=3 0.8512; e=8: 0.8494